In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q /content/drive/MyDrive/Comys_Hackathon5.zip -d /content/Comys_Hackathon5

In [3]:
!sudo pip3 install keras
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import cv2
import os
import random
import shutil
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras import applications, models, layers, callbacks, optimizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing import image_dataset_from_directory

In [6]:
def build_embedding_model(img_size=(224, 224, 3), embedding_dim=128):
    base_cnn = tf.keras.applications.DenseNet121(
        input_shape=img_size,
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    base_cnn.trainable = False

    input_layer = layers.Input(shape=img_size)
    x = base_cnn(input_layer)
    x = layers.Dense(embedding_dim)(x)
    x = tf.keras.layers.UnitNormalization()(x)

    return models.Model(input_layer, x)


siamese_model = tf.keras.models.load_model("siamese_best_model.keras")
embedding_model = build_embedding_model()


def load_and_preprocess_image(img_path, img_size=(224, 224)):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=img_size)
    img = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    return img


def index_taskB_data(root_path):
    identity_dict = {}
    for identity in os.listdir(root_path):
        identity_path = os.path.join(root_path, identity)
        if not os.path.isdir(identity_path):
            continue

        clean_images = []
        distorted_images = []

        for item in os.listdir(identity_path):
            item_path = os.path.join(identity_path, item)
            if os.path.isfile(item_path) and item.lower().endswith(('.jpg', '.jpeg', '.png')):
                clean_images.append(item_path)
            elif os.path.isdir(item_path) and item == 'distortion':
                for dist_file in os.listdir(item_path):
                    dist_path = os.path.join(item_path, dist_file)
                    if dist_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        distorted_images.append(dist_path)

        if clean_images and distorted_images:
            identity_dict[identity] = {
                'clean': clean_images,
                'distorted': distorted_images
            }

    return identity_dict


def create_reference_embeddings(ref_index, embedding_model):
    ref_embeddings = {}
    for identity, data in tqdm(ref_index.items(), desc="Creating reference embeddings"):
        embs = []
        for path in data["clean"]:
            img = load_and_preprocess_image(path)
            img = np.expand_dims(img, axis=0)
            emb = embedding_model.predict(img, verbose=0)[0]
            emb = emb / (np.linalg.norm(emb) + 1e-10)
            embs.append(emb)
        ref_embeddings[identity] = np.array(embs)
    return ref_embeddings


def evaluate_task(test_data_path, threshold=0.5):
    index = index_taskB_data(test_data_path)
    ref_embeddings = create_reference_embeddings(index, embedding_model)

    y_true = []
    y_pred = []

    for identity, data in tqdm(index.items(), desc="Evaluating test data"):
        for dist_path in data["distorted"]:
            img = load_and_preprocess_image(dist_path)
            img = np.expand_dims(img, axis=0)
            dist_emb = embedding_model.predict(img, verbose=0)[0]
            dist_emb = dist_emb / (np.linalg.norm(dist_emb) + 1e-10)

            best_id = None
            best_sim = -1

            for ref_id, ref_embs in ref_embeddings.items():
                ref_embs = ref_embs / (np.linalg.norm(ref_embs, axis=1, keepdims=True) + 1e-10)
                sims = np.dot(ref_embs, dist_emb)
                max_sim = np.max(sims)
                if max_sim > best_sim:
                    best_sim = max_sim
                    best_id = ref_id

            match_label = 1 if best_id == identity and best_sim >= threshold else 0
            y_true.append(1)
            y_pred.append(match_label)

    acc = accuracy_score(y_true, y_pred) * 100
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0) * 100
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100

    print("\nEvaluation on Test Set:")
    print("Top-1 Accuracy: {:.2f} %".format(acc))
    print("Macro Precision: {:.2f} %".format(precision))
    print("Macro Recall: {:.2f} %".format(recall))
    print("Macro F1 Score: {:.2f} %".format(f1))

evaluate_task("/content/Comys_Hackathon5/Comys_Hackathon5/Task_B/val")

Evaluating test data: 100%|██████████| 250/250 [04:57<00:00,  1.19s/it]


Evaluation on Test Set:
Top-1 Accuracy: 58.97 %
Macro Precision: 50.00 %
Macro Recall: 29.49 %
Macro F1 Score: 37.10 %
